In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import Sequential, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.metrics import Precision, Recall
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
cc_df = pd.read_csv('/kaggle/input/airline-passangers-booking-data/Customer_comment.csv')
cc_df.head()

In [ ]:
cc_df.info()
print("Shape of dataset:", cc_df.shape)


Objective 1:***Data cleaning***

# Finding nulls + number of rows

In [ ]:
# Drop unnecessary columns from Customer_comment.csv
cols_to_drop_cc = [
    'Unnamed: 0',# irrelevent 
    'flight_number',# irrelevent 
    'scheduled_departure_date',# irrelevent 
    'entity',# irrelevent to Goal
    'ques_verbatim_text',# Constant Irrelevent values 
    'verbatim_text'# Repeated unrefined Column (transform text replaces it)
]

cc_df = cc_df.drop(columns=cols_to_drop_cc)


print("Remaining columns after dropping:")
print(cc_df.columns.tolist())


In [ ]:

cc_df.isnull().sum()

In [ ]:
cc_df['loyalty_program_level'] = cc_df['loyalty_program_level'].fillna(
    cc_df['loyalty_program_level'].mode()[0]
)
cc_df.dropna(subset=['transformed_text'], inplace=True)
cc_df.isnull().sum()


* Loyality Program nulls less than 25 percent of the non null rows and catagorical so mode is the best fill option
* Transformed Text Column removed Null rows Due to insignificance and low number of rows

In [ ]:
cc_df.head()

In [ ]:
pbd_df = pd.read_csv('/kaggle/input/airline-passangers-booking-data/Passanger_booking_data.csv')
pbd_df.head()

In [ ]:
pbd_df.isnull().sum()

In [ ]:
pbd_df.info()

In [ ]:

pbd_df = pbd_df.drop_duplicates()


In [ ]:
sds_df = pd.read_csv('/kaggle/input/airline-passangers-booking-data/Survey data_Inflight Satisfaction Score.csv')
sds_df.head()

In [ ]:

sds_df.info()
sds_df.isnull().sum()

In [ ]:
cols_to_drop = [
    'cabin_name',                     # 50 percent missing and  2 uniques values
    'question_text',                 # Repeated Values
    'flight_number',                 # Irrelevent
    'record_locator',                # Irrelevent id 
    'arrival_delay_group',           # Redundant (we keep minutes)
    'driver_sub_group1', 'driver_sub_group2',  # Repeated Values
    'number_of_legs',                # Constant value in most cases
    'equipment_type_code', 'ua_uax', # Irrelevent codes
    'departure_gate', 'arrival_gate',# Not relevant to satisfaction
    'hub_spoke',                     # Operational label
]

sds_df = sds_df.drop(columns=cols_to_drop)
print("Remaining columns:", sds_df.columns.tolist())

In [ ]:

sds_df.isnull().sum()

In [ ]:
sds_df.dropna(subset=['satisfaction_type'], inplace=True)
sds_df['media_provider'] = sds_df['media_provider'].fillna(sds_df['media_provider'].mode()[0])
sds_df['loyalty_program_level'] = sds_df['loyalty_program_level'].fillna(sds_df['loyalty_program_level'].mode()[0])
sds_df['entity'] = sds_df['entity'].fillna(sds_df['entity'].mode()[0])


* satisfaction_type important and could be dropped better than filled
* media_provider small percentage of nulls filled using mode
* loyalty_program_level Not as important categorical filled with mode
* entity only three missing values filled with mode 

In [ ]:
sds_df.isnull().sum()

In [ ]:
sds_df.head()

*POINT 2* 

In [ ]:

!pip install vaderSentiment

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


In [ ]:
scrpd_df= pd.read_csv('/kaggle/input/yamosahel/AirlineScrappedReview_Cleaned.csv')
scrpd_df.head()


In [ ]:
scrpd_df.info()
scrpd_df.isnull().sum()

In [ ]:
# Drop irrelevant or highly missing columns
cols_to_drop = [
    'Passanger_Name', #irrelevant
    'Flying_Date', #80 percent missing
    'Layover_Route',#86 percent missing
    'Start_Location', 'End_Location',# 411 or more unique values Irrelevant for prediction 
    'Review_title', #repeated
     'Start_Latitude','Start_Longitude','End_Latitude','End_Longitude','Start_Address','End_Address' # Location irrelevent from airline
]
scrpd_df = scrpd_df.drop(columns=cols_to_drop)

print("Remaining columns after dropping:")
print(scrpd_df.columns.tolist())
scrpd_df = scrpd_df.dropna(subset=['Route'])
scrpd_df.isnull().sum()

In [ ]:
# Find the mode for Traveller_type, excluding "Unknown"
traveller_mode = scrpd_df[scrpd_df['Traveller_Type'] != 'Unknown']['Traveller_Type'].mode()[0]

# Replace "Unknown" with the mode
scrpd_df['Traveller_Type'] = scrpd_df['Traveller_Type'].replace('Unknown', traveller_mode)
# Find the mode (most frequent value) of Class column, excluding "Unknown"
class_mode = scrpd_df[scrpd_df['Class'] != 'Unknown']['Class'].mode()[0]

# Replace "Unknown" with the mode
scrpd_df['Class'] = scrpd_df['Class'].replace('Unknown', class_mode)

In [ ]:
scrpd_df.isnull().sum()

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# Example usage
sample_text = "The flight was amazing and the staff were wonderful!"
print(analyzer.polarity_scores(sample_text))

In [ ]:
def get_sentiment(text):
    if pd.isnull(text):
        return 0
    return analyzer.polarity_scores(str(text))['compound']

scrpd_df['sentiment_score'] = scrpd_df['Review_content'].apply(get_sentiment)

def label_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

scrpd_df['sentiment_label'] = scrpd_df['sentiment_score'].apply(label_sentiment)

scrpd_df.head(100)

In [ ]:
completed_bookings = pbd_df[pbd_df['booking_complete'] == 1]

# Get the top 10 most popular routes
top_routes = completed_bookings['route'].value_counts().head(10)

# Display results
print("Top 10 Most Popular Flight Routes (with completed bookings):")
print(top_routes)


In [ ]:
completed_bookings = pbd_df[pbd_df['booking_complete'] == 1]
top_routes = completed_bookings['route'].value_counts().head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top_routes.values, y=top_routes.index, palette='viridis')
plt.title('Top 10 Most Popular Flight Routes (Completed Bookings Only)')
plt.xlabel('Number of Completed Bookings')
plt.ylabel('Route')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

The top route, AKLKUL (Auckland to Kuala Lumpur), dominates, with over 40% more completed bookings than the second-place route. Furthermore, PEN (Penang) appears as either an origin or destination in five of the top ten routes, establishing it as a critical regional hub.

Objective 2 : ***Data engineering questions***

In [ ]:
completed = pbd_df[pbd_df['booking_complete'] == 1]

# Get the distribution of bookings across flight hours
hour_distribution = completed['flight_hour'].value_counts().sort_index()

# Display the distribution
print("Distribution of Completed Bookings by Flight Hour:")
print(hour_distribution)

In [ ]:
completed = pbd_df[pbd_df['booking_complete'] == 1]

plt.figure(figsize=(10,6))
sns.histplot(completed['flight_hour'], bins=24, kde=True, color='skyblue')
plt.title('Distribution of Completed Bookings Across Flight Hours')
plt.xlabel('Flight Hour')
plt.ylabel('Number of Completed Bookings')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Completed bookings show two clear peaks, corresponding to standard travel times:

* Morning Peak (6 AM - 9 AM): Demand builds quickly, peaking at 9 AM (507 bookings).

* Afternoon Peak (12 PM - 2 PM): A secondary period of high activity, with bookings remaining robust through 2 PM.

The demand drops sharply after 3 PM, hitting a trough during late evening hours (7 PM - 10 PM), suggesting fewer late-night flights or a preference against them.

In [ ]:
avg_ratings = scrpd_df.groupby(['Traveller_Type', 'Class'])['Rating'].mean().reset_index()
avg_ratings = avg_ratings.sort_values('Rating', ascending=False)
avg_ratings

In [ ]:
highest = avg_ratings.iloc[0]
lowest = avg_ratings.iloc[-1]

print(f"Highest rated combination: {highest['Traveller_Type']} - {highest['Class']} with average rating {highest['Rating']:.2f}")
print(f"Lowest rated combination: {lowest['Traveller_Type']} - {lowest['Class']} with average rating {lowest['Rating']:.2f}")



plt.figure(figsize=(10,5))
sns.barplot(data=avg_ratings, x='Traveller_Type', y='Rating', hue='Class', palette='viridis')
plt.title('Average Rating by Traveller Type and Class')
plt.ylabel('Average Rating')
plt.xlabel('Traveller Type')
plt.legend(title='Class')
plt.show()

1. Solo Leisure Satisfaction is High: Solo Leisure travelers are generally the most satisfied, especially in premium classes, suggesting that the core product and service (which is often optimized for individual travelers) is effective for this segment.

2. Business Traveler Dissatisfaction: Business travelers consistently report the lowest satisfaction across the board, regardless of class (Business and Economy both average 3.46). This suggests a fundamental failure to meet the needs of this segment. Business travelers prioritize efficiency, punctuality, and productivity (e.g., Wi-Fi, quiet environment), and the current service delivery is missing the mark.

3. Family/Premium Paradox: Family Leisure in First Class (4.23) is rated significantly lower than Solo Leisure in Economy Class (4.56), indicating that the First Class offering does not effectively cater to the complex needs of families (e.g., space, entertainment, specialized assistance).

Objective 3: ***Model training***

In [ ]:
# Define Target (Y) and Features (X)
# ==============================

# Create a new binary target column 'satisfaction'
scrpd_df['satisfaction'] = np.where(scrpd_df['Rating'] >= 5, 1, 0)

# Define feature columns
X = scrpd_df[['Traveller_Type', 'Class', 'Verified', 'Route', 'sentiment_score']]
y = scrpd_df['satisfaction']


In this block we define the features that we will use in order to train our models and the target variable , both are under the names X,Y. moreover our targeted column 'satisfaction' has been derived from the column called "Rating"


In [ ]:
scrpd_df.head(5)

We just double checked here in order to see how our dataSet looked,just for further calarification

In [ ]:
#  Encode Categorical Variables
# ==============================
# Convert categorical features to dummy variables
X = pd.get_dummies(X, columns=['Traveller_Type', 'Class', 'Verified', 'Route'], drop_first=True)
X = X.astype('float32')
# ==============================

Since all of our input features exlucding the **"sentiment_score"** is of categorical values we must encode them in order to train our model with it. we then turn their values to be of float32 in order to assure consistency and efficeny between different models as some models such as gradient boosting expect the features to be of float

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1️⃣ Split first (to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2️⃣ Identify the numeric column(s)
num_cols = ['sentiment_score']

# 3️⃣ Initialize MinMaxScaler (scales to [0, 1] by default)
scaler = MinMaxScaler()

# 4️⃣ Fit on training data, transform both train & test
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


Here we first split our dataset to be 80% of training data and 20% for testing, and since we have all features of categorical values between 0,1 we decided to scale down the sentiment_score to be between range 0 and 1
so we ensure all values of our features are of the same propotional range as this is the best practice when training neural networks.

In [ ]:
from tensorflow.keras import layers, Sequential, Input
from tensorflow.keras.metrics import Precision, Recall

model = Sequential([
    Input(shape=(X_train.shape[1],)),          # Input layer (one neuron per feature)
    layers.Dense(16, activation='relu'),       # First hidden layer with 16 neurons
    layers.Dense(8, activation='relu'),        # Second hidden layer with 8 neurons
    layers.Dense(8, activation='relu'),        # Third hidden layer with 8 neurons
    layers.Dense(8, activation='relu'),        # Fourth hidden layer with 8 neurons
    layers.Dense(1, activation='sigmoid')      # Output layer for binary classification
])

# Display model summary
model.summary()

# ==============================================================
# 3. Compile the Model
# ==============================================================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)

# ==============================================================
# 4. Train the Neural Network Model
# ==============================================================

from tensorflow.keras.callbacks import EarlyStopping

# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=50,               # Maximum number of epochs
    batch_size=32,           # Number of samples per gradient update
    validation_split=0.2,    # 80% training / 20% validation split
    verbose=1,
    callbacks=[early_stop]
)

# ==============================================================
# 5. Evaluate Model Performance
# ==============================================================

train_loss, train_acc, train_prec, train_rec = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc, test_prec, test_rec = model.evaluate(X_test, y_test, verbose=0)

print("\n================== Model Evaluation ==================")
print(f"Training Accuracy:  {train_acc:.4f}")
print(f"Testing Accuracy:   {test_acc:.4f}")
print(f"Training Precision: {train_prec:.4f}")
print(f"Testing Precision:  {test_prec:.4f}")
print(f"Training Recall:    {train_rec:.4f}")
print(f"Testing Recall:     {test_rec:.4f}")
print("=======================================================")

Here we build and train a simple neural network for binary classification 🧠.
We chose a neural network as our baseline model because it’s powerful enough to learn complex patterns but also known for having unstable accuracy, making it a great reference point for improvement. It’s flexible, widely used, and gives us a solid foundation to compare future models against.

Our model learns through two hidden layers and predicts with a sigmoid output — perfect for yes/no outcomes.
We track accuracy, precision, and recall to get a full picture of performance, and use early stopping to keep it from overfitting.

Finally, we test it on both training and test data to see if it truly learned or just memorized, giving us a balanced and trustworthy baseline. 🚀

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score



# ==============================================================================
# 1. Hyperparameter Tuning (Fixing Underfitting)
# ==============================================================================
# Loosen the constraints to allow the model to predict the positive class
rf_model_tuned = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15,             
    min_samples_leaf=5,       
    class_weight='balanced',  
    random_state=42
)

# Train the tuned model on the inputed features
rf_model_tuned.fit(X_train, y_train)

# Predict on the correctly scaled test data
y_pred_rf_tuned = rf_model_tuned.predict(X_test)
# ... rest of your evaluation code ...







Here, we define and train our Random Forest model using the selected hyperparameters.
After training, in the following 2 codeblocks we will check how well it performs on both the training and validation sets.

We use Random Forest because it handles our mix of categorical features and one numeric feature (sentiment_score) well, and it’s effective for binary classification tasks like predicting if a customer is satisfied or not.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ==============================
# 🔹 Training & Validation Performance Plots
# ==============================
plt.figure(figsize=(6, 4))
sns.barplot(x=list(train_metrics.keys()), y=list(train_metrics.values()), palette='viridis')
plt.title("Random Forest - Training Performance", fontsize=13, weight='bold')
plt.ylabel("Score")
plt.ylim(0, 1)
for i, v in enumerate(train_metrics.values()):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center', fontsize=10)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.barplot(x=list(val_metrics.keys()), y=list(val_metrics.values()), palette='magma')
plt.title("Random Forest - Validation (Cross-Validation) Performance", fontsize=13, weight='bold')
plt.ylabel("Score")
plt.ylim(0, 1)
for i, v in enumerate(val_metrics.values()):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center', fontsize=10)
plt.tight_layout()
plt.show()

# ==============================
# 🔹 Model Evaluation on Test Data
# ==============================
def evaluate_model(y_true, y_pred, model_name):
    """Prints key metrics and a detailed classification report."""
    print(f"\n--- {model_name} ---")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.3f}")
    print(f"F1 Score : {f1_score(y_true, y_pred):.3f}")
    print("\nDetailed Report:\n", classification_report(y_true, y_pred))

# Evaluate Tuned Random Forest on Test Set
evaluate_model(y_test, y_pred_rf_tuned, "Tuned Random Forest (Test Data)")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score



# ==============================
# 🔹 Calculate Metrics (Using Tuned Model Predictions)
# ==============================
# UPDATED to use y_pred_rf_tuned
accuracy = accuracy_score(y_test, y_pred_rf_tuned)
precision = precision_score(y_test, y_pred_rf_tuned)
recall = recall_score(y_test, y_pred_rf_tuned)
f1 = f1_score(y_test, y_pred_rf_tuned)


# ==============================
# 🔹 Confusion Matrix Heatmap
# ==============================
# UPDATED to use y_pred_rf_tuned
cm = confusion_matrix(y_test, y_pred_rf_tuned)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix - Tuned Random Forest")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
# Note: Assuming 0 is Dissatisfied (318 support) and 1 is Satisfied (244 support)
plt.xticks([0.5, 1.5], ['Dissatisfied (0)', 'Satisfied (1)'])
plt.yticks([0.5, 1.5], ['Dissatisfied (0)', 'Satisfied (1)'], rotation=0)
plt.show()


# ==============================
# 🔹 Model Performance Metrics Bar Plot
# ==============================
metrics = {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}

plt.figure(figsize=(6,4))
# Using the recommended color palette for accessibility and professionalism
sns.barplot(x=list(metrics.keys()), y=list(metrics.values()), palette='viridis') 
plt.title("Tuned Random Forest Model Performance")
plt.ylabel("Score")
plt.ylim(0, 1)
for i, v in enumerate(list(metrics.values())):
    # Display the score value on the bar
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center', fontsize=10)
plt.show()

The past 2 code blocks checks how our Random Forest model performs on the training, validation, and test sets.
We look at the results to make sure the model isn’t overfitting and can generalize well to new data.
This step is important because it helps confirm that our model actually learns useful patterns and not just memorizes the training data.

In [ ]:
import pandas as pd
import numpy as np

def predict_satisfaction(
    traveller_type,
    travel_class,
    verified,
    route,
    sentiment_score,
    rf_model=rf_model_tuned,
    scaler=scaler,
    feature_template=X
):
    """
    Predicts passenger satisfaction (Satisfied / Not Satisfied)
    based on raw input features using the trained Random Forest model.
    """
    
    # Step 1: Create a DataFrame from the single input
    input_data = pd.DataFrame([{
        'Traveller_Type': traveller_type,
        'Class': travel_class,
        'Verified': verified,
        'Route': route,
        'sentiment_score': sentiment_score
    }])
    
    # Step 2: Apply the same one-hot encoding as training
    input_encoded = pd.get_dummies(
        input_data,
        columns=['Traveller_Type', 'Class', 'Verified', 'Route'],
        drop_first=True
    )
    
    # Step 3: Align with training features (ensure same columns & order)
    input_encoded = input_encoded.reindex(columns=feature_template.columns, fill_value=0)
    

    prediction = rf_model.predict(input_encoded)[0]
    
    # Step 6: Convert numeric output to human-readable label
    if prediction == 1:
        return "Satisfied 😀"
    else:
        return "Not Satisfied 😞"


This code block let us predict if a passenger is Satisfied 😀 or Not Satisfied 😞 based on their details like traveller type, class, verification, route, and sentiment score.
It uses the same encoding and trained Random Forest model to keep everything consistent with how the model was trained.

This step is important because it shows how the model can be used in a real scenario to predict customer satisfaction from new inputs.

In [ ]:
result = predict_satisfaction(
    traveller_type="Business",
    travel_class="Economy Class",
    verified="Trip Verified",
    route="London Heathrow to Arlanda Stockholm",
    sentiment_score=0.9716
)

print("Prediction:The Customer is", result)

Here we use the prediction function we defined in the last code block and put it to use by creating a random data of the input features and pass it down to the function and then print the result.
This step is important to test whether our prediction function works or not

In [ ]:
!pip install shap lime
import shap
import lime
import lime.lime_tabular

Here we imports the nesscary libraries in order to use XAI modules we will be using in the following blocks

In [ ]:
# Initialize the SHAP explainer for your trained Random Forest model
explainer = shap.TreeExplainer(rf_model_tuned)

# Compute SHAP values for the test set
shap_values = explainer.shap_values(X_test)

# Display summary plot (global importance)
shap.summary_plot(
    shap_values[1],  # Class = 1 (satisfied)
    X_test,
    feature_names=X.columns,
    plot_type="bar"
)

# More detailed SHAP summary (beeswarm plot)
shap.summary_plot(
    shap_values[1],
    X_test,
    feature_names=X.columns
)

Here we use SHAP to see which features are driving our Random Forest predictions.

The bar plot shows which features matter the most for predicting satisfied customers.

The beeswarm plot digs deeper, showing how different feature values push predictions toward satisfied or not satisfied.

This helps us understand the model’s decisions and see what really influences customer satisfaction

In [ ]:
# Choose a sample index to explain
i = 0  # you can change this to any row index

shap.initjs()

# Force plot for local explanation
shap.force_plot(
    explainer.expected_value[1],  # 1 = satisfied
    shap_values[1][i],
    X_test.iloc[i],
    feature_names=X.columns
)

Local Prediction Explanation with SHAP

This part shows why the model made a specific prediction for one passenger.

The force plot breaks down how each feature pushed the prediction toward Satisfied 😀 or Not Satisfied 😞 for that particular example.

It’s useful to see the individual reasoning behind the model’s decision, not just overall feature importance.

In [ ]:
# Ensure correct formats
X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df = pd.DataFrame(X_test, columns=X.columns)

# LIME explainer
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train_df.values,
    feature_names=X.columns,
    class_names=['Not Satisfied', 'Satisfied'],
    discretize_continuous=True
)

# Explain instance
i = 0
exp = explainer_lime.explain_instance(
    X_test_df.values[i],
    rf_model_tuned.predict_proba,
    num_features=6
)
exp.show_in_notebook()

Here we use LIME to see why the model predicted Satisfied 😀 or Not Satisfied 😞 for a specific passenger.

It highlights the top features that influenced that individual prediction.

We use LIME because it provides local, human-readable explanations, showing how the model arrives at decisions for each instance, which complements global methods like SHAP.

This helps make the model’s predictions more transparent and trustworthy, especially for individual cases